In [ ]:
# ============================================================
# 08_MARL_vs_Baselines.ipynb – Proper Comparison + Ablation
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

ROOT = Path("..")
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X = np.load(DATA / "X_fused.npy")
y = np.load(DATA / "y.npy")
thin = np.load(DATA / "thin.npy")

X_train, X_test, y_train, y_test, thin_train, thin_test = train_test_split(
    X, y, thin, test_size=0.25, random_state=42, stratify=y
)

def evaluate_classifier(name, y_true, y_prob, y_pred, thin_mask):
    return {
        "Model": name,
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Thin-file Approval": (y_pred[thin_mask] == 0).mean() if thin_mask.sum() > 0 else 0
    }

results = []

# ----- Traditional Baselines -----
for name, model in [
    ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ("Random Forest", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
    ("XGBoost", xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42))
]:
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    results.append(evaluate_classifier(name, y_test, prob, pred, thin_test==1))

# ----- Load your trained MARL (from notebook 07) -----
# (You need to also load the specialized agents if you want full inference)
# For now we show the structure – replace with real loading

print("Traditional baselines done.")
res_df = pd.DataFrame(results)
print(res_df.round(3))
res_df.to_csv(RESULTS / "comparison_table.csv", index=False)